# 05 — Uplift Evaluation: Qini, AUUC & Uplift@K

## Objective
Evaluate uplift models using metrics designed for **incremental response**, not ordinary classification accuracy.

This notebook calculates:
- Qini curves
- AUUC
- Uplift@10%
- Uplift@20%
- Uplift@30%
- model comparison
- uplift distributions

In [ ]:
from pathlib import Path
import sys, json, warnings
warnings.filterwarnings("ignore")

# Find the project root so the notebook works whether launched from the project root
# or from inside the notebooks directory.
_here = Path.cwd().resolve()
_candidates = [_here, *_here.parents]
ROOT = next((p for p in _candidates if (p / "src" / "config.py").exists()), _here)
if not (ROOT / "src" / "config.py").exists():
    raise FileNotFoundError("Could not locate project root containing src/config.py")

sys.path.insert(0, str(ROOT))
print("Project root:", ROOT)

In [ ]:
import numpy as np
import pandas as pd
from src.evaluation import uplift_metrics, uplift_curve
from src.data_utils import load_data, prepare
from src.feature_engineering import make_preprocessor
from src.uplift_models import train_tlearner, predict_tlearner, train_class_transformation
from src.config import DATA_PATH, RANDOM_SEED, TEST_SIZE
from sklearn.model_selection import train_test_split

df = prepare(load_data(DATA_PATH))
idx_train, idx_test = train_test_split(
    np.arange(len(df)), test_size=TEST_SIZE,
    random_state=RANDOM_SEED, stratify=df["segment"]
)
train = df.iloc[idx_train].reset_index(drop=True)
test = df.iloc[idx_test].reset_index(drop=True)
pre = make_preprocessor()
X_train = pre.fit_transform(train)
X_test = pre.transform(test)

results = []
curves = {}

for arm, slug in [("Mens E-Mail","mens"), ("Womens E-Mail","womens")]:
    mt, mc = train_tlearner(X_train, train.segment.to_numpy(), train.conversion.to_numpy(), arm)
    pt, pc = predict_tlearner(mt, mc, X_test)
    uplift_tm = pt - pc
    m_tm, c_tm = uplift_metrics(test.conversion, test.segment, uplift_tm, arm)
    results.append({"comparison":f"{arm} vs Control", "model":"Two-Model", **m_tm})
    curves[f"{slug}_two_model"] = uplift_curve(test.conversion, test.segment, uplift_tm, arm)

    ct, _ = train_class_transformation(X_train, train.segment.to_numpy(), train.conversion.to_numpy(), arm)
    uplift_ct = ct.predict(X_test)
    m_ct, c_ct = uplift_metrics(test.conversion, test.segment, uplift_ct, arm)
    results.append({"comparison":f"{arm} vs Control", "model":"Class Transformation", **m_ct})
    curves[f"{slug}_class_transform"] = uplift_curve(test.conversion, test.segment, uplift_ct, arm)

# Any Email
t_train_any = np.where(train.segment == "No E-Mail", "No E-Mail", "Any Email")
t_test_any = np.where(test.segment == "No E-Mail", "No E-Mail", "Any Email")
mt, mc = train_tlearner(X_train, t_train_any, train.conversion.to_numpy(), "Any Email")
pt, pc = predict_tlearner(mt, mc, X_test)
uplift_any = pt - pc
m_any, c_any = uplift_metrics(test.conversion, t_test_any, uplift_any, "Any Email")
results.append({"comparison":"Any Email vs Control", "model":"Two-Model", **m_any})
curves["any_two_model"] = uplift_curve(test.conversion, t_test_any, uplift_any, "Any Email")

comparison = pd.DataFrame(results).sort_values("auuc", ascending=False).reset_index(drop=True)
display(comparison.style.format({
    "auuc":"{:.6f}", "qini":"{:.6f}",
    "uplift_at_10":"{:.4%}", "uplift_at_20":"{:.4%}", "uplift_at_30":"{:.4%}"
}))
comparison.to_csv(ROOT / "artifacts" / "metrics" / "notebook_05_model_comparison.csv", index=False)

## 1. Qini / cumulative incremental-response curves

In [ ]:
import matplotlib.pyplot as plt

fig_dir = ROOT / "artifacts" / "figures"
fig_dir.mkdir(parents=True, exist_ok=True)

for name, curve in curves.items():
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(curve["fraction"], curve["incremental_response"], label=name)
    ax.axhline(0, linestyle="--")
    ax.set_title(f"Uplift Curve — {name}")
    ax.set_xlabel("Fraction of customers targeted")
    ax.set_ylabel("Cumulative incremental response")
    ax.legend()
    fig.tight_layout()
    fig.savefig(fig_dir / f"notebook_05_{name}.png", dpi=160)
    plt.show()

## 2. Uplift distribution

In [ ]:
# Refit the strongest comparison from the table so the distribution is directly inspectable.
best = comparison.iloc[0]
print("Best evaluated model:", best.to_dict())

arm = "Mens E-Mail" if "Mens" in best["comparison"] else ("Womens E-Mail" if "Womens" in best["comparison"] else "Any Email")
slug = "mens" if arm == "Mens E-Mail" else ("womens" if arm == "Womens E-Mail" else "any")

if arm == "Any Email":
    train_t = np.where(train.segment == "No E-Mail", "No E-Mail", "Any Email")
    test_t = np.where(test.segment == "No E-Mail", "No E-Mail", "Any Email")
else:
    train_t = train.segment.to_numpy()
    test_t = test.segment.to_numpy()

if best["model"] == "Two-Model":
    mt, mc = train_tlearner(X_train, train_t, train.conversion.to_numpy(), arm)
    pt, pc = predict_tlearner(mt, mc, X_test)
    best_uplift = pt - pc
else:
    ct, _ = train_class_transformation(X_train, train_t, train.conversion.to_numpy(), arm)
    best_uplift = ct.predict(X_test)

fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(best_uplift, bins=40)
ax.axvline(0, linestyle="--", label="Zero uplift")
ax.set_title(f"Estimated Uplift Distribution — {best['model']} — {arm}")
ax.set_xlabel("Estimated incremental conversion probability")
ax.set_ylabel("Customers")
ax.legend()
fig.tight_layout()
fig.savefig(fig_dir / "notebook_05_best_uplift_distribution.png", dpi=160)
plt.show()

## Interpretation
- **AUUC** summarizes the area under the cumulative uplift curve.
- **Qini** here is the maximum cumulative incremental-response value from the implementation.
- **Uplift@K** asks how much incremental response is observed/estimated among the highest-ranked K% of customers.

These are more relevant to targeting quality than ordinary accuracy because the business objective is **incremental response caused by treatment**.